# 02b. 일관성 점검 (사실 확인)

**판정하지 않는다.** CoT 안의 서로 다른 부분이 같은 사실을 같게 적고 있는지, 데이터셋에 중복·편중이 있는지 **건수와 샘플**만 확인한다.
품질 기준 판정은 03 단계(판정 함수)에서 한다.

| # | 항목 | 비교 대상 |
|---|---|---|
| 1 | 환경 | git이 원본을 추적하지 않는지 |
| 2 | 상품 등급 | cot3에 적힌 상품별 등급 vs 상품 JSON `risk_grade` |
| 3 | 성향별 허용 등급 | 투자성향(cot1/cot2) × cot2 허용 등급 집합 |
| 4 | 추천 상품명 | answer 본문 vs `product_names` |
| 5 | 메타 vs 본문 | 메타 age·gender vs question 본문 |
| 6 | 고령자 고위험 | 70대 이상 × 1~2등급 추천 |
| 7 | 중복 | question 완전·유사 중복 |
| 8 | 추천 편중 | 상품별 추천 횟수 |
| 9 | 문장 완결 | answer 끝 문장 부호 |

추출 함수는 `src/extract.py`, 상품명 매칭은 D10 확정 규칙(정확 → 정규화 → 보험 모상품명)을 따른다.

In [ ]:
# 라이브러리와 프로젝트 함수 불러오기
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.load_cot import load_cot, load_products
from src.extract import (build_product_index, match_product, product_grade, parse_allowed_grades,
                         grade_after_name, propensity_in, extract_age_decade, extract_gender,
                         meta_age_match, quoted_strings, ends_with_punct, norm_full,
                         near_duplicate_groups, PROPENSITY)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

In [ ]:
# 데이터 읽기: CoT 전체(Train+Valid), 상품 JSON(Training 폴더, D1)
cot = load_cot()
prod = load_products()
idx = build_product_index(prod)
print("CoT:", cot.shape, "/ 상품:", prod.shape)

In [ ]:
# 추천 상품명을 한 행에 하나씩 펼치고 D10 규칙으로 상품 JSON에 매칭
rec = (cot[["split", "query_group", "category", "cot_id", "age", "product_names"]]
       .explode("product_names").dropna(subset=["product_names"])
       .rename(columns={"product_names": "rec_name"}).reset_index(drop=True))
m = rec["rec_name"].map(lambda n: match_product(n, idx))
rec["match_stage"] = m.str[0]
rec["keys"] = m.str[1]
rec["json_grade"] = rec["keys"].map(lambda k: product_grade(k, idx))
print("추천 상품명 수:", len(rec))
rec["match_stage"].value_counts()

## 1. 환경: git이 원본을 추적하지 않는지

In [ ]:
# data/raw/ 아래 파일이 .gitignore로 제외되는지 확인 (제외되면 규칙과 파일명이 출력됨)
r = subprocess.run(["git", "-C", str(ROOT), "check-ignore", "-v", "data/raw/Training/02.라벨링데이터/TL_2.소비자.zip"],
                   capture_output=True, text=True, encoding="utf-8")
print("check-ignore:", r.stdout.strip() or "(제외되지 않음)")

# git이 추적 중이거나 추가 대상으로 보는 파일 중 data/ 아래 파일 수
r = subprocess.run(["git", "-C", str(ROOT), "-c", "core.quotepath=false", "status", "--porcelain", "-uall"],
                   capture_output=True, text=True, encoding="utf-8")
tracked_data = [l for l in r.stdout.splitlines() if " data/" in l]
print("git status에 나온 data/ 파일 수:", len(tracked_data))

# 노트북 출력 제거 필터(nbstripout) 설정 확인
print(open(ROOT / ".gitattributes", encoding="utf-8").read())

## 2. cot3에 적힌 상품별 등급 vs 상품 JSON `risk_grade` (증권)

- 샘플 확인 결과 cot2에는 상품명이 나오지 않아(0%), 상품별 등급은 **cot3**에서 뽑는다.
- 범위 표현("N등급 이상/이하", "N~M등급", "N·M등급")은 제외한다.
- 추출 실패는 "상품명 뒤에 등급 표현 자체가 없음"과 "있지만 추출 실패(범위 표현만 있음)"로 나눈다.

In [ ]:
# 증권 추천 상품별로 cot3에서 등급을 뽑는다
sec = rec[rec["category"] == "증권"].merge(cot[["cot_id", "split", "cot3", "product_names"]], on=["cot_id", "split"])
res = sec.apply(lambda r: grade_after_name(r["cot3"], r["rec_name"], r["product_names"]), axis=1)
sec["cot3_status"] = res.str[0]
sec["cot3_grade"] = res.str[1]
sec["cot3_status"].value_counts().to_frame("건수")

In [ ]:
# 추출 성공 건에서 JSON 등급과 비교
# 비교 불가: 상품 JSON 매칭 실패 또는 이름 중복으로 등급이 여러 개인 경우
ok = sec[sec["cot3_status"] == "추출"].copy()
ok["비교"] = np.select(
    [ok["json_grade"].isna(), ok["cot3_grade"] == ok["json_grade"]],
    ["비교 불가(JSON 등급 없음)", "일치"], default="불일치")
print(ok["비교"].value_counts())
print()
# 불일치 건: cot3 등급 × JSON 등급 교차표
mis = ok[ok["비교"] == "불일치"]
pd.crosstab(mis["cot3_grade"], mis["json_grade"], margins=True)

In [ ]:
# 불일치 샘플 10건: 상품명 뒤 구간의 앞 120자를 함께 본다
def after_name(text, name, n=120):
    i = str(text).find(name)
    return str(text)[i + len(name): i + len(name) + n] if i >= 0 else ""

s = mis.sample(min(10, len(mis)), random_state=0)
for _, r in s.iterrows():
    print(f"[{r['split']} cot_id={r['cot_id']}] {r['rec_name']}")
    print(f"   cot3 등급={r['cot3_grade']:.0f} / JSON 등급={r['json_grade']:.0f}")
    print("   상품명 뒤:", after_name(r["cot3"], r["rec_name"]).replace("\n", " "))

In [ ]:
# 추출 실패 샘플: '등급 표현 없음' 5건, '있지만 추출 실패' 5건 (상품명 뒤 150자)
for status in ["등급 표현 없음", "있지만 추출 실패"]:
    sub = sec[sec["cot3_status"] == status]
    print("=" * 100)
    print(status, f"({len(sub)}건)")
    for _, r in sub.sample(min(5, len(sub)), random_state=0).iterrows():
        print(f"[cot_id={r['cot_id']}] {r['rec_name']}")
        print("   상품명 뒤:", after_name(r["cot3"], r["rec_name"], 150).replace("\n", " "))

## 3. 투자성향별 cot2 허용 등급 표기

- 성향은 cot1 기준과 cot2 기준을 나눠 뽑는다 (증권_03은 cot2에 성향이 나온다).
- 성향 명칭이 2개 이상이면 '복수', 없으면 '없음'으로 둔다.

In [ ]:
# 증권 CoT만 대상 (보험 cot2에는 등급 표현이 거의 없음: 02 노트북 6-3)
sc = cot[cot["category"] == "증권"].copy()
sc["성향_cot1"] = sc["cot1"].map(propensity_in)
sc["성향_cot2"] = sc["cot2"].map(propensity_in)
parsed = sc["cot2"].map(parse_allowed_grades)
sc["허용등급"] = parsed.str[0].map(lambda s: "-".join(map(str, sorted(s))) if s else "추출 실패")
sc["허용등급_출처"] = parsed.str[1]
print(sc["허용등급_출처"].value_counts())

In [ ]:
# cot1 성향 × 허용 등급 집합
pd.crosstab(sc["성향_cot1"], sc["허용등급"], margins=True)

In [ ]:
# cot2 성향 × 허용 등급 집합
pd.crosstab(sc["성향_cot2"], sc["허용등급"], margins=True)

In [ ]:
# 같은 성향(cot1 또는 cot2에서 하나만 나온 경우)인데 허용 등급 집합이 몇 종류인지
sc["성향_대표"] = np.where(sc["성향_cot1"].isin(["없음", "복수"]), sc["성향_cot2"], sc["성향_cot1"])
one = sc[~sc["성향_대표"].isin(["없음", "복수"]) & (sc["허용등급"] != "추출 실패")]
summary = one.groupby("성향_대표")["허용등급"].agg(
    건수="size", 허용등급_종류수="nunique", 최빈값=lambda s: s.value_counts().index[0],
    최빈값_비율=lambda s: round(s.value_counts(normalize=True).iloc[0] * 100, 1))
summary

In [ ]:
# 증권_03 중 cot1에 성향이 없는 건 → cot2에 성향이 있는지
s3 = sc[sc["query_group"] == "증권_03.상품 비교 추천형"]
no1 = s3[s3["성향_cot1"] == "없음"]
print("증권_03 전체:", len(s3), "/ cot1에 성향 없음:", len(no1))
no1["성향_cot2"].value_counts().to_frame("cot2 성향")

## 4. answer 속 상품명 vs `product_names`

두 가지 방법으로 본다.
- (A) answer에서 큰따옴표 안의 문자열을 뽑아 `product_names`와 비교
- (B) `product_names`의 각 상품명(정규화)이 answer 본문(정규화)에 포함되는지

In [ ]:
# (A) 따옴표 추출
ans = cot[["split", "query_group", "cot_id", "answer", "product_names"]].copy()
ans["quoted"] = ans["answer"].map(quoted_strings)
ans["list_norm"] = ans["product_names"].map(lambda L: {norm_full(x) for x in L})
ans["quoted_norm"] = ans["quoted"].map(lambda L: {norm_full(x) for x in L})
# answer 따옴표에만 있는 이름 (목록에 없음)
ans["only_answer"] = ans.apply(lambda r: r["quoted_norm"] - r["list_norm"], axis=1)
# 목록에는 있으나 answer 따옴표에 없는 이름
ans["only_list"] = ans.apply(lambda r: r["list_norm"] - r["quoted_norm"], axis=1)
print("따옴표가 하나도 없는 answer:", (ans["quoted"].map(len) == 0).sum())
print("answer 따옴표에만 있는 이름이 1개 이상인 CoT:", (ans["only_answer"].map(len) > 0).sum())
print("목록에만 있는 이름이 1개 이상인 CoT:", (ans["only_list"].map(len) > 0).sum())

In [ ]:
# (B) product_names 각 상품명이 answer 본문에 포함되는지 (둘 다 정규화)
ans["answer_norm"] = ans["answer"].map(norm_full)
ans["list_missing"] = ans.apply(lambda r: [x for x in r["product_names"] if norm_full(x) not in r["answer_norm"]], axis=1)
ans["n_missing"] = ans["list_missing"].map(len)
print("목록 상품 중 answer 본문에 없는 상품이 1개 이상인 CoT:", (ans["n_missing"] > 0).sum())
pd.crosstab(ans["query_group"], ans["n_missing"])

In [ ]:
# 샘플: (A) answer 따옴표에만 있는 이름 5건, (B) 본문에 없는 목록 상품 5건
print("=== (A) answer 따옴표에만 있는 이름 ===")
for _, r in ans[ans["only_answer"].map(len) > 0].sample(5, random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] 따옴표 원문:", [q for q in r["quoted"] if norm_full(q) in r["only_answer"]][:2])
print()
print("=== (B) answer 본문에 없는 목록 상품 ===")
sub = ans[ans["n_missing"] > 0]
for _, r in sub.sample(min(5, len(sub)), random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] 없는 상품:", r["list_missing"][:2])

## 5. 메타 age·gender vs question 본문

In [ ]:
# 본문에서 연령대·성별 추출 후 메타와 비교
q = cot[["split", "query_group", "category", "cot_id", "age", "gender", "question"]].copy()
q["본문_연령대"] = q["question"].map(extract_age_decade)
q["연령_비교"] = [meta_age_match(a, d) for a, d in zip(q["age"], q["본문_연령대"])]
q["본문_성별"] = q["question"].map(extract_gender)
q["성별_비교"] = np.select([q["본문_성별"].isna(), q["본문_성별"] == "복수", q["본문_성별"] == q["gender"]],
                         ["본문에 없음", "본문에 복수", "일치"], default="불일치")
print(pd.crosstab(q["query_group"], q["연령_비교"], margins=True))
print()
print(pd.crosstab(q["query_group"], q["성별_비교"], margins=True))

In [ ]:
# 불일치 샘플 (연령 5건, 성별 5건): 본문에서 표현 주변 60자
import re
def around(text, pat, n=30):
    m = re.search(pat, str(text))
    return str(text)[max(0, m.start() - n): m.end() + n] if m else ""

print("=== 연령 불일치 ===")
for _, r in q[q["연령_비교"] == "불일치"].sample(5, random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] 메타={r['age']} / 본문:", around(r["question"], r"[1-9]0\s*대|\d{2}\s*(?:세|살)"))
print("=== 성별 불일치 ===")
sub = q[q["성별_비교"] == "불일치"]
for _, r in sub.sample(min(5, len(sub)), random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] 메타={r['gender']} / 본문:", around(r["question"], r"남성|남자|여성|여자"))

## 6. 70대 이상 고객 CoT 중 1~2등급 상품 추천 (증권)

- 메타 age가 `70대`, `70대 이상`, `80대 이상`인 CoT.
- 등급은 상품 JSON 기준. 보험 상품에는 등급이 없어 증권만 본다.

In [ ]:
# 고령 메타 연령 × 증권 추천 상품의 JSON 등급
OLD = ["70대", "70대 이상", "80대 이상"]
old = rec[(rec["category"] == "증권") & rec["age"].isin(OLD)].copy()
print("대상 CoT 수:", old["cot_id"].nunique(), "/ 추천 상품 수:", len(old))
print(pd.crosstab(old["age"], old["json_grade"].fillna(-1).astype(int).replace(-1, "등급 없음/불명"), margins=True))

hi = old[old["json_grade"].isin([1, 2])]
print()
print("1~2등급 상품이 1개 이상 추천된 CoT 수:", hi["cot_id"].nunique())

In [ ]:
# 샘플 10건: 메타 연령, 성향(cot1/cot2), cot2 허용 등급, 추천 상품과 등급
hs = hi.merge(sc[["cot_id", "성향_cot1", "성향_cot2", "허용등급"]], on="cot_id", how="left")
hs.drop_duplicates("cot_id").sample(min(10, hi["cot_id"].nunique()), random_state=0)[
    ["cot_id", "age", "성향_cot1", "성향_cot2", "허용등급", "rec_name", "json_grade"]]

## 7. question 중복

- 완전 중복: 앞뒤 공백 제거·연속 공백 통일 후 문장이 똑같은 경우
- 유사 중복: 글자 2~3그램 TF-IDF 코사인 유사도 0.9 이상을 연결한 묶음 (500행 블록 단위 계산)

In [ ]:
# 완전 중복
q["q_norm"] = q["question"].str.strip().str.replace(r"\s+", " ", regex=True)
grp = q.groupby("q_norm")
dup_groups = grp.filter(lambda g: len(g) > 1)
print("완전 중복 묶음 수:", dup_groups["q_norm"].nunique(), "/ 포함 행 수:", len(dup_groups))
print("Train↔Valid에 걸친 완전 중복 묶음 수:", dup_groups.groupby("q_norm")["split"].nunique().gt(1).sum())

In [ ]:
# 유사 중복 (시간이 조금 걸린다)
q["near_group"] = near_duplicate_groups(q["question"].tolist(), threshold=0.9, block=500)
sizes = q.groupby("near_group").size()
multi = sizes[sizes > 1]
print("유사 중복 묶음 수(2건 이상):", len(multi), "/ 포함 행 수:", multi.sum())
print("최대 묶음 크기:", multi.max() if len(multi) else 0, "/ 10건 이상 묶음 수:", (multi >= 10).sum())
print()
print("묶음 크기 분포:")
print(multi.value_counts().sort_index().rename("묶음 수").to_frame().T)

In [ ]:
# 묶음이 어느 split에 걸쳐 있는지: Train만 / Valid만 / Train↔Valid
splits = q[q["near_group"].isin(multi.index)].groupby("near_group")["split"].agg(lambda s: "↔".join(sorted(set(s))))
splits.value_counts().rename("묶음 수").to_frame()

In [ ]:
# 가장 큰 유사 중복 묶음의 질문 3개 앞 120자
if len(multi) == 0:
    print("유사 중복 묶음이 없다.")
else:
    big = multi.idxmax()
    for t in q.loc[q["near_group"] == big, "question"].head(3):
        print("-", t[:120])

In [ ]:
# 참고: 임계값을 낮추면 묶음이 얼마나 생기는지 (0.9 결과를 해석할 때 비교용, 판정 아님)
for th in [0.8, 0.7]:
    lab = near_duplicate_groups(q["question"].tolist(), threshold=th, block=500)
    s = pd.Series(lab).value_counts()
    s = s[s > 1]
    print(f"유사도 {th} 이상: 묶음 {len(s)}개, 포함 행 {s.sum()}, 최대 크기 {s.max() if len(s) else 0}")

# 각 질문의 '가장 비슷한 다른 질문' 유사도 분포 (블록 계산)
from sklearn.feature_extraction.text import TfidfVectorizer
X = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)).fit_transform(q["question"])
best = []
for s0 in range(0, X.shape[0], 500):
    sim = (X[s0:s0 + 500] @ X.T).toarray()
    for i in range(sim.shape[0]):
        sim[i, s0 + i] = 0          # 자기 자신 제외
    best.append(sim.max(axis=1))
best = np.concatenate(best)
pd.Series(best).describe(percentiles=[.5, .9, .99]).round(3)

## 8. 상품별 추천 횟수

In [ ]:
# 매칭된 상품 키 기준 추천 횟수
# 애매한 경우 처리: 이름 중복으로 후보가 여러 개면 대표 이름(display_name) 단위로 센다
tbl = idx["table"]
rec["display_name"] = rec["keys"].map(lambda k: tbl.loc[k[0], "display_name"] if k else None)
cnt = rec.dropna(subset=["display_name"]).groupby(["category", "display_name"]).size().rename("추천 횟수")
cnt.sort_values(ascending=False).head(20).to_frame()

In [ ]:
# 한 번도 추천되지 않은 상품 수 (상품 JSON 10,000개 기준)
used_keys = set(k for ks in rec["keys"] for k in ks)
tbl2 = tbl.assign(추천됨=tbl.index.isin(used_keys))
out = tbl2.groupby("domain")["추천됨"].agg(전체="size", 추천됨="sum")
out["추천 안 됨"] = out["전체"] - out["추천됨"]
print(out)
print()
print("주의: 매칭 실패(판정불가) 추천명", (rec["keys"].map(len) == 0).sum(), "건은 어떤 상품에도 연결되지 않아, 일부 상품이 '추천 안 됨'으로 세어졌을 수 있다.")

In [ ]:
# 상품 분류별 추천 안 된 비율
g = tbl2.groupby(["domain", "product_group"])["추천됨"].agg(전체="size", 추천됨="sum")
g["추천 안 된 비율(%)"] = ((1 - g["추천됨"] / g["전체"]) * 100).round(1)
g

## 9. answer가 문장 부호 없이 끝나는 건

In [ ]:
# 끝 공백·닫는 따옴표/괄호를 건너뛴 마지막 글자가 . ! ? … 。 가 아닌 건
ans["ends_ok"] = ans["answer"].map(ends_with_punct)
print("문장 부호 없이 끝나는 answer:", (~ans["ends_ok"]).sum())
print(pd.crosstab(ans["query_group"], ans["ends_ok"]))

In [ ]:
# 샘플 10건: 끝 60자
for _, r in ans[~ans["ends_ok"]].sample(min(10, (~ans["ends_ok"]).sum()), random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] ...{r['answer'][-60:]!r}")

# 추가 확인 (2026-11-10)

판정 없이 사실만 확인한다.

| # | 항목 |
|---|---|
| A1 | [기준 3 핵심] 추천 상품의 JSON 등급이 cot2 허용 등급 집합 안에 있는지 |
| A2 | 성향별 허용 등급 집합 전체 분포 (질문유형별) |
| A3 | '등급 N' 표기 인식 추가 후, 수정 전 '등급 표현 없음' 건 재집계 |
| A4 | 고령 1~2등급 추천 CoT의 성향 분포와 연령 고려 표현 |
| A5 | 성별 불일치 63건에서 cot1~answer가 따르는 성별 |
| A6 | answer에 없는 목록 상품의 cot3 서술 |

## A1. 추천 상품 JSON 등급 ∈ cot2 허용 등급 집합 (증권)

In [ ]:
# CoT별 허용 등급 집합을 추천 상품 행에 붙인다
sc["허용집합"] = parsed.str[0]
a1 = rec[rec["category"] == "증권"].merge(sc[["cot_id", "허용집합", "허용등급", "허용등급_출처", "성향_대표", "cot2"]], on="cot_id")

# 애매한 경우 처리: 허용 집합 추출 실패, JSON 등급 없음(매칭 실패·등급 복수)은 안/밖을 정하지 않고 따로 센다
def in_range(r):
    if r["허용집합"] is None:
        return "허용 집합 추출 실패"
    if pd.isna(r["json_grade"]):
        return "JSON 등급 없음"
    return "범위 안" if int(r["json_grade"]) in r["허용집합"] else "범위 밖"

a1["허용범위"] = a1.apply(in_range, axis=1)
print(a1["허용범위"].value_counts())
print()
print(pd.crosstab(a1["query_group"], a1["허용범위"], margins=True))
print()
# 허용 집합을 어떤 근거로 뽑았는지(parse_allowed_grades의 근거 종류)별로 나눠 본다
# '까지 해석', '이상·이하 해석', '단일 등급만'은 해석이나 부분 정보가 들어간 집합이다
pd.crosstab(a1["허용등급_출처"], a1["허용범위"], margins=True)

In [ ]:
# 범위 밖: 성향 × (JSON 등급) 교차표
out_ = a1[a1["허용범위"] == "범위 밖"]
print("범위 밖 상품이 있는 CoT 수:", out_["cot_id"].nunique())
pd.crosstab([out_["성향_대표"], out_["허용등급"]], out_["json_grade"].astype(int))

In [ ]:
# 범위 밖 샘플 10건: cot2에서 '등급'이 들어간 문장만 보여 준다
import re
def grade_sents(t, n=220):
    s = [x for x in re.split(r"(?<=[.!?])\s+", str(t)) if "등급" in x]
    return " / ".join(s)[:n]

for _, r in out_.sample(min(10, len(out_)), random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] 성향={r['성향_대표']} / 허용={r['허용등급']} / JSON={int(r['json_grade'])}등급 / {r['rec_name']}")
    print("   cot2:", grade_sents(r["cot2"]))

In [ ]:
# 허용 집합 추출 실패 건의 cot2 앞부분 3건 (실패 이유 확인용)
fail = a1[a1["허용범위"] == "허용 집합 추출 실패"].drop_duplicates("cot_id")
print("허용 집합 추출 실패 CoT 수:", len(fail))
for _, r in fail.head(3).iterrows():
    print(f"[cot_id={r['cot_id']}]", str(r["cot2"])[:200])

## A2. 성향별 허용 등급 집합 분포 (질문유형별)

In [ ]:
# 성향(성향_대표)마다 허용 등급 집합 × 질문유형 건수표
for p in ["위험중립형", "안정추구형", "적극투자형", "공격투자형", "안정형"]:
    sub = sc[sc["성향_대표"] == p]
    print("=" * 80)
    print(f"{p} ({len(sub)}건)")
    print(pd.crosstab(sub["허용등급"], sub["query_group"], margins=True, margins_name="합계")
            .sort_values("합계", ascending=False))

## A3. '등급 N' 표기 인식 추가 후 재집계

`grade_after_name(..., suffix=False)`는 수정 전 방식('N등급'만 인식), 기본값 `suffix=True`는 '위험등급 2', '등급 3' 같은 표기도 인식한다.
위 2번 셀의 결과(`sec`)는 이미 수정 후 방식으로 계산된 값이다.

In [ ]:
# 수정 전 방식으로 다시 계산해 수정 후 상태와 비교
old = sec.apply(lambda r: grade_after_name(r["cot3"], r["rec_name"], r["product_names"], suffix=False), axis=1)
sec["cot3_status_old"] = old.str[0]
print(pd.crosstab(sec["cot3_status_old"], sec["cot3_status"], margins=True))

In [ ]:
# 수정 전 '등급 표현 없음'이었던 건 → 수정 후 상태와, 새로 추출된 등급의 JSON 일치 여부
was_none = sec[sec["cot3_status_old"] == "등급 표현 없음"]
print("수정 전 '등급 표현 없음':", len(was_none))
print(was_none["cot3_status"].value_counts())
newly = was_none[was_none["cot3_status"] == "추출"]
cmp_ = np.select([newly["json_grade"].isna(), newly["cot3_grade"] == newly["json_grade"]],
                 ["JSON 등급 없음", "일치"], default="불일치")
print()
print("새로 추출된 건의 JSON 비교:", pd.Series(cmp_).value_counts().to_dict())

In [ ]:
# 수정 후에도 '등급 표현 없음'인 건 샘플 3건 (상품명 뒤 150자)
for _, r in sec[sec["cot3_status"] == "등급 표현 없음"].sample(3, random_state=1).iterrows():
    print(f"[cot_id={r['cot_id']}] {r['rec_name']}")
    print("   상품명 뒤:", after_name(r["cot3"], r["rec_name"], 150).replace("\n", " "))

## A4. 고령(70대·80대 이상) CoT 중 1~2등급 추천 156건 상세

In [ ]:
# 156건과 나머지 고령 CoT(1~2등급 추천 없음)의 성향 분포
old_ids = set(old_cot := rec.loc[(rec["category"] == "증권") & rec["age"].isin(OLD), "cot_id"])
hi_ids = set(hi["cot_id"])
g = sc[sc["cot_id"].isin(old_ids)].copy()
g["구분"] = np.where(g["cot_id"].isin(hi_ids), "1~2등급 추천 있음", "1~2등급 추천 없음")
print(pd.crosstab(g["성향_대표"], g["구분"], margins=True))
print()
print("cot1 기준:")
print(pd.crosstab(g["성향_cot1"], g["구분"]))

In [ ]:
# 연령 고려 표현이 있는 건수 (필드별, 단어별)
# 애매한 경우 처리: '나이'는 '나이에 맞는'처럼 연령 고려가 아닌 문맥도 잡힐 수 있다. 문맥은 구분하지 않는다.
AGE_WORDS = ["고령", "연세", "노후", "나이", "은퇴", "연령", "노년", "시니어"]
# (sc는 cot에서 복사한 것이라 cot1~answer 컬럼이 이미 있다)
rows = []
for f in ["cot1", "cot2", "cot3", "answer"]:
    for w in AGE_WORDS:
        has = g[f].astype(str).str.contains(w)
        rows.append({"필드": f, "단어": w,
                     "1~2등급 추천 있음": has[g["구분"] == "1~2등급 추천 있음"].sum(),
                     "1~2등급 추천 없음": has[g["구분"] == "1~2등급 추천 없음"].sum()})
tbl_age = pd.DataFrame(rows).pivot(index="단어", columns="필드")
tbl_age

In [ ]:
# 네 필드 중 어디든 연령 고려 표현(위 8개 단어)이 하나라도 있는 CoT 수
pat = "|".join(AGE_WORDS)
g["연령표현_어디든"] = g[["cot1", "cot2", "cot3", "answer"]].astype(str).agg(" ".join, axis=1).str.contains(pat)
g["연령표현_사용자4개"] = g[["cot1", "cot2", "cot3", "answer"]].astype(str).agg(" ".join, axis=1).str.contains("고령|연세|노후|나이")
g.groupby("구분")[["연령표현_어디든", "연령표현_사용자4개"]].agg(["sum", "size"])

## A5. 성별 불일치 63건: 각 필드가 따르는 성별

In [ ]:
# 메타(여)와 본문(남)이 다른 63건에서 cot1·cot2·cot3·answer의 성별 언급
mm = q[q["성별_비교"] == "불일치"].merge(cot[["cot_id", "cot1", "cot2", "cot3", "answer"]], on="cot_id")
def follow(text_gender, body, meta):
    if text_gender is None:
        return "언급 없음"
    if text_gender == "복수":
        return "복수"
    return "본문 따름" if text_gender == body else ("메타 따름" if text_gender == meta else "기타")

res5 = pd.DataFrame({f: [follow(extract_gender(t), b, m) for t, b, m in zip(mm[f], mm["본문_성별"], mm["gender"])]
                     for f in ["cot1", "cot2", "cot3", "answer"]})
res5.apply(pd.Series.value_counts).fillna(0).astype(int)

In [ ]:
# cot1에서 성별을 언급한 문장 예시 3건
for _, r in mm.head(3).iterrows():
    print(f"[cot_id={r['cot_id']}] 메타={r['gender']} / 본문={r['본문_성별']} / cot1:", around(r["cot1"], r"남성|남자|여성|여자"))

## A6. answer에 없는 목록 상품 110건: cot3 서술

In [ ]:
# answer에 없는 목록 상품을 한 행씩 펼치고, cot3에서 그 상품명 뒤 구간을 본다
from src.extract import _find_all
miss = ans[ans["n_missing"] > 0][["cot_id", "query_group", "list_missing", "product_names"]].explode("list_missing")
miss = miss.merge(cot[["cot_id", "cot3"]], on="cot_id")

def cot3_seg(r, n=200):
    sp = _find_all(str(r["cot3"]), r["list_missing"])
    return str(r["cot3"])[sp[0][1]: sp[0][1] + n] if sp else None

miss["cot3_seg"] = miss.apply(cot3_seg, axis=1)
print("answer에 없는 목록 상품 수:", len(miss), "/ CoT 수:", miss["cot_id"].nunique())
print("cot3에 그 상품명이 있는 경우:", miss["cot3_seg"].notna().sum())

In [ ]:
# cot3 구간에 '제외·부적합' 계열 표현이 있는지 (단어별 건수)
# 애매한 경우 처리: '다만', '반면'은 단순 비교일 수도 있어 따로 센다. 문맥은 판단하지 않는다.
EXCL = {"제외": r"제외", "적합하지 않/부적합": r"적합하지\s*않|부적합", "부합하지 않": r"부합하지\s*않",
        "맞지 않": r"맞지\s*않", "추천하지 않/드리지 않": r"추천하지\s*않|추천드리지\s*않|권하지\s*않",
        "어렵": r"어렵", "다만": r"다만", "반면": r"반면"}
seg = miss["cot3_seg"].fillna("")
excl_tbl = pd.Series({k: seg.str.contains(p, regex=True).sum() for k, p in EXCL.items()}, name="건수")
print(excl_tbl)
strict = r"제외|적합하지\s*않|부적합|부합하지\s*않|맞지\s*않|추천하지\s*않|추천드리지\s*않|권하지\s*않"
print()
print("'다만·반면·어렵'을 뺀 제외 계열 표현이 있는 상품 수:", seg.str.contains(strict, regex=True).sum())

In [ ]:
# 샘플 10건: cot3에서 상품명 뒤 200자
for _, r in miss.sample(min(10, len(miss)), random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] {r['list_missing']}")
    seg_txt = r["cot3_seg"] if isinstance(r["cot3_seg"], str) else "(cot3에 상품명 없음)"
    print("   cot3:", seg_txt.replace("\n", " "))

# 추가 확인 2 (2026-11-11)

판정 없이 사실만 확인한다.

| # | 항목 |
|---|---|
| B1 | 연령대별 증권 추천 상품 중 1~2등급 비율 (고령 vs 그 외, 성향별) |
| B2 | 허용 범위 밖 33건 CoT의 예외 사유 서술 유무 |
| B3 | answer에서 추천 상품마다 숫자 위험등급이 언급되는지 |
| B4 | (D12 확정용) 보험_01 추천 상품 중 변액보험·외화보험 |

## B1. 연령대별 1~2등급 추천 비율 (증권, 상품 JSON 등급 기준)

In [ ]:
# 증권 추천 상품 중 JSON 등급이 있는 건만 대상
# 애매한 경우 처리: JSON 등급이 없는 추천(매칭 실패·등급 복수)은 분모에서 뺀다
b1 = rec[(rec["category"] == "증권") & rec["json_grade"].notna()].merge(sc[["cot_id", "성향_대표"]], on="cot_id")
b1["1~2등급"] = b1["json_grade"].isin([1, 2])
b1["연령그룹"] = np.where(b1["age"].isin(OLD), "70대 이상(70대·80대 이상)", "그 외")

def rate_tbl(df, by):
    t = df.groupby(by)["1~2등급"].agg(추천상품수="size", 등급1_2="sum")
    t["비율(%)"] = (t["등급1_2"] / t["추천상품수"] * 100).round(1)
    return t

print(rate_tbl(b1, "age"))
print()
print(rate_tbl(b1, "연령그룹"))

In [ ]:
# 성향별로 고령 vs 그 외 비교
t = rate_tbl(b1, ["성향_대표", "연령그룹"])["비율(%)"].unstack()
t.join(rate_tbl(b1, ["성향_대표", "연령그룹"])["추천상품수"].unstack(), rsuffix="_n")

## B2. 허용 범위 밖 CoT의 예외 사유 서술

방법: cot2·cot3에서 예외 사유를 나타낼 수 있는 표현을 찾는다.
- 고객 요청 계열: `요청`, `명시`, `원하`, `희망`, `선호`
- 예외·전환 계열: `예외`, `불구하고`, `그러나`, `다만`, `벗어`, `초과`

애매한 경우 처리: 단어가 있다고 예외 사유를 서술한 것으로 확정할 수 없다(예: '선호'는 일반 서술에도 나온다).
그래서 **고객 요청 계열과 예외·전환 계열이 같은 문장에 함께 나오는 경우**만 '서술 있음'으로 분류하고, 각 CoT의 해당 문장을 전부 출력해 눈으로 확인할 수 있게 한다.

In [ ]:
REQ = r"요청|명시|원하|희망|선호"
EXC = r"예외|불구하고|그러나|다만|벗어|초과"

def exception_sents(text):
    sents = re.split(r"(?<=[.!?])\s+", str(text))
    return [s for s in sents if re.search(REQ, s) and re.search(EXC, s)]

b2 = out_.drop_duplicates("cot_id")[["cot_id", "query_group", "성향_대표", "허용등급"]].merge(
    cot[["cot_id", "cot2", "cot3"]], on="cot_id")
b2["예외문장_cot2"] = b2["cot2"].map(exception_sents)
b2["예외문장_cot3"] = b2["cot3"].map(exception_sents)
b2["분류"] = np.where((b2["예외문장_cot2"].map(len) + b2["예외문장_cot3"].map(len)) > 0, "서술 있음", "서술 없음")
print("범위 밖 CoT 수:", len(b2))
print(pd.crosstab(b2["성향_대표"], b2["분류"], margins=True))

In [ ]:
# '서술 있음' CoT의 해당 문장 (앞 160자)
for _, r in b2[b2["분류"] == "서술 있음"].iterrows():
    print(f"[cot_id={r['cot_id']}] {r['성향_대표']} / 허용 {r['허용등급']}")
    for s in (r["예외문장_cot2"] + r["예외문장_cot3"])[:2]:
        print("   -", s[:160])

In [ ]:
# '서술 없음' CoT: cot2에서 '등급'이 들어간 문장 (앞 160자)
for _, r in b2[b2["분류"] == "서술 없음"].iterrows():
    print(f"[cot_id={r['cot_id']}] {r['성향_대표']} / 허용 {r['허용등급']} / cot2: {grade_sents(r['cot2'], 160)}")

## B3. answer에서 추천 상품마다 숫자 위험등급이 언급되는지

방법: cot3와 같은 함수(`grade_after_name`)를 answer에 적용한다. 함께 나열된 상품은 같은 등급 표현을 공유한다.

In [ ]:
b3 = rec.merge(cot[["cot_id", "answer", "product_names"]], on="cot_id")
res = b3.apply(lambda r: grade_after_name(r["answer"], r["rec_name"], r["product_names"]), axis=1)
b3["ans_status"] = res.str[0]
b3["ans_grade"] = res.str[1]
print(pd.crosstab(b3["query_group"], b3["ans_status"], margins=True))

In [ ]:
# answer에 등급이 추출된 증권 상품: JSON 등급과 일치 여부
s3 = b3[(b3["category"] == "증권") & (b3["ans_status"] == "추출")]
cmp3 = np.select([s3["json_grade"].isna(), s3["ans_grade"] == s3["json_grade"]], ["JSON 등급 없음", "일치"], default="불일치")
print(pd.Series(cmp3).value_counts())

# CoT 단위: 추천 상품 모두에 등급이 있음 / 일부만 / 하나도 없음 (상품명 없음은 제외하고 계산)
per = b3[b3["ans_status"] != "상품명 없음"].groupby(["category", "cot_id"])["ans_status"].agg(
    lambda s: "모두 있음" if (s == "추출").all() else ("하나도 없음" if (s != "추출").all() else "일부만"))
print()
print(per.groupby(level=0).value_counts().unstack(fill_value=0))

## B4. (D12 확정용) 보험_01 추천 상품 중 변액보험·외화보험

- 변액보험: 상품 JSON `product_type == '변액'` 또는 상품명에 '변액'
- 외화보험: 상품명·상품 설명 필드에 `달러|외화|USD|미화|위안|엔화|유로`

애매한 경우 처리: 설명 필드에 '달러' 등이 나오더라도 외화 상품이 아닌 문맥(예: 비교 설명)일 수 있다. 어느 필드에서 걸렸는지 함께 표시한다.

In [ ]:
b4 = rec[rec["query_group"] == "보험_01.연령별 및 생애주기 저축성 상품 추천형"].copy()
b4 = b4[b4["keys"].map(len) > 0]   # 매칭된 추천만
b4["key"] = b4["keys"].str[0]      # 이름 중복 시 첫 후보 (애매한 경우: 후보끼리 유형이 다를 수 있음)
info = tbl.loc[b4["key"]].reset_index()
FX = r"달러|외화|USD|미화|위안|엔화|유로"
desc_cols = ["product_full_name", "mother_product_name", "rider_type", "product_features", "tax_type",
             "minimum_guarantee", "refund_condition", "payment_conditions1"]
b4["product_type"] = info["product_type"].values
b4["변액(유형)"] = b4["product_type"].eq("변액").values
b4["변액(이름)"] = b4["rec_name"].str.contains("변액").values
fx_hit = info[desc_cols].astype(str).apply(lambda col: col.str.contains(FX, regex=True))
b4["외화_걸린필드"] = fx_hit.apply(lambda r: ",".join(r.index[r]), axis=1).values
b4["외화(이름)"] = b4["rec_name"].str.contains(FX, regex=True).values
print("보험_01 추천 상품(매칭됨):", len(b4), "/ CoT:", b4["cot_id"].nunique())
print(b4["product_type"].value_counts())
print()
summary4 = pd.DataFrame({
    "추천 건수": [b4["변액(유형)"].sum(), b4["변액(이름)"].sum(), b4["외화(이름)"].sum(), (b4["외화_걸린필드"] != "").sum()],
    "고유 상품 수": [b4.loc[b4["변액(유형)"], "key"].nunique(), b4.loc[b4["변액(이름)"], "key"].nunique(),
                b4.loc[b4["외화(이름)"], "key"].nunique(), b4.loc[b4["외화_걸린필드"] != "", "key"].nunique()],
    "CoT 수": [b4.loc[b4["변액(유형)"], "cot_id"].nunique(), b4.loc[b4["변액(이름)"], "cot_id"].nunique(),
              b4.loc[b4["외화(이름)"], "cot_id"].nunique(), b4.loc[b4["외화_걸린필드"] != "", "cot_id"].nunique()],
}, index=["변액: product_type", "변액: 상품명", "외화: 상품명", "외화: 이름·설명 필드"])
summary4

In [ ]:
# 샘플 5개씩: 변액 / 외화 (고유 상품 기준)
print("=== 변액 ===")
v = b4[b4["변액(유형)"] | b4["변액(이름)"]].drop_duplicates("key")
for _, r in v.head(5).iterrows():
    print(f"- {r['rec_name']} (product_type={r['product_type']})")
print("=== 외화 ===")
f = b4[(b4["외화_걸린필드"] != "") | b4["외화(이름)"]].drop_duplicates("key")
for _, r in f.head(5).iterrows():
    print(f"- {r['rec_name']} (걸린 필드: {r['외화_걸린필드'] or '상품명'})")

# 추가 확인 3 (2026-11-11)

판정 없이 사실만 확인한다. (C4는 D19 정의를 그대로 적용한 집계)

| # | 항목 |
|---|---|
| C1 | 파서 보완('N등급, N등급, N등급' 목록) 후 범위 밖 재집계 |
| C2 | cot1과 cot2의 투자성향이 다른 건 |
| C3 | 70대 이상 안정형 중 1~2등급 추천 건의 원문 |
| C4 | D19 기준 answer 위험 설명 충족 여부 |
| C5 | 메타 age '50대 이상', '60대 이상'의 의미 |
| C6 | 범위 밖 CoT 분류용 표 |

※ 위쪽 A1·A2·B1·B2 셀도 보완된 파서로 다시 계산된 값이다.

## C1. 파서 보완 후 범위 밖 재집계

In [ ]:
# 수정 전(D13 기록): 범위 밖 83건(CoT 33), 허용 집합 추출 실패 73건(CoT 29)
print("범위 밖 추천 상품:", (a1["허용범위"] == "범위 밖").sum(), "/ CoT:", a1.loc[a1["허용범위"] == "범위 밖", "cot_id"].nunique())
print("허용 집합 추출 실패:", (a1["허용범위"] == "허용 집합 추출 실패").sum(),
      "/ CoT:", a1.loc[a1["허용범위"] == "허용 집합 추출 실패", "cot_id"].nunique())
print()
print(sc["허용등급_출처"].value_counts())

## C2. cot1과 cot2의 투자성향이 다른 건 (증권)

In [ ]:
# 둘 다 성향 명칭이 하나씩 나온 경우만 비교 (없음·복수는 비교에서 뺀다)
both = sc[sc["성향_cot1"].isin(PROPENSITY) & sc["성향_cot2"].isin(PROPENSITY)]
diff = both[both["성향_cot1"] != both["성향_cot2"]]
print("cot1·cot2 모두 성향 1개:", len(both), "/ 서로 다름:", len(diff))
print()
print(pd.crosstab(diff["성향_cot1"], diff["성향_cot2"], margins=True))
print()
print(diff["query_group"].value_counts())

In [ ]:
# 샘플 5건: cot1과 cot2에서 성향 명칭이 들어간 문장
def prop_sent(t, n=150):
    s = [x for x in re.split(r"(?<=[.!?])\s+", str(t)) if any(p in x for p in PROPENSITY)]
    return " / ".join(s)[:n]

for _, r in diff.sample(min(5, len(diff)), random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] cot1={r['성향_cot1']} / cot2={r['성향_cot2']}")
    print("   cot1:", prop_sent(r["cot1"]))
    print("   cot2:", prop_sent(r["cot2"]))

## C3. 70대 이상 안정형 중 1~2등급 추천 건

In [ ]:
c3 = b1[(b1["성향_대표"] == "안정형") & (b1["연령그룹"] != "그 외") & b1["1~2등급"]]
print("해당 추천 상품 수:", len(c3), "/ CoT 수:", c3["cot_id"].nunique())
for _, r in c3.merge(sc[["cot_id", "성향_cot1", "성향_cot2", "cot2"]], on="cot_id").iterrows():
    gset, kind, raw = parse_allowed_grades(r["cot2"], with_text=True)
    print("=" * 90)
    print(f"[cot_id={r['cot_id']}] {r['age']} / cot1={r['성향_cot1']} / cot2={r['성향_cot2']}")
    print(f"   추천: {r['rec_name']} → JSON {int(r['json_grade'])}등급")
    print(f"   허용 집합: {sorted(gset) if gset else None} ({kind}) / 원문: {raw}")
    print("   cot2 등급 문장:", grade_sents(r["cot2"], 300))

## C4. D19 기준 answer 위험 설명 충족 여부 (증권)

D19: 추천 상품마다 숫자 등급 또는 공식 명칭(매우높은위험~매우낮은위험) 중 하나가 answer에 있으면 충족.

애매한 경우 처리: answer에 상품명이 없는 추천(상품명 없음)은 그 상품에 대한 설명을 찾을 수 없으므로 **미충족**으로 센다. 몇 건인지 따로 표시한다.

In [ ]:
from src.extract import risk_mention_after_name
c4 = b3[b3["category"] == "증권"].copy()
c4["mention"] = c4.apply(lambda r: risk_mention_after_name(r["answer"], r["rec_name"], r["product_names"]), axis=1)
c4["충족"] = c4["mention"].isin(["숫자 등급", "공식 명칭", "둘 다"])
print(c4["mention"].value_counts())
print()
print("추천 상품 기준 충족:", c4["충족"].sum(), "/", len(c4), f"({c4['충족'].mean()*100:.1f}%)")

In [ ]:
# CoT 단위: 전부 충족 / 일부 / 전무
cot_lvl = c4.groupby("cot_id")["충족"].agg(lambda s: "전부 충족" if s.all() else ("전무" if not s.any() else "일부"))
print(cot_lvl.value_counts())
print()
# 비교: B3(숫자 등급만) 기준
print(pd.crosstab(c4.drop_duplicates("cot_id").set_index("cot_id").loc[cot_lvl.index, "query_group"], cot_lvl, margins=True))
print()
print("'상품명 없음' 추천이 있어 미충족이 된 CoT 수:", c4.loc[c4["mention"] == "상품명 없음", "cot_id"].nunique())

## C5. 메타 age '50대 이상', '60대 이상'

B1 표의 `50대 이상`(4건), `60대 이상`(3건)은 **추천 상품 수**다. 이 값을 가진 증권 CoT 수는 아래와 같다.
메타 age에는 `50대`와 `50대 이상`, `60대`와 `60대 이상`이 따로 있다(02 노트북 4단계). 두 값이 어떻게 다른지는 문서에 설명이 없어 **불확실**하다.

In [ ]:
print(sc["age"].value_counts().sort_index())
print()
for a in ["50대 이상", "60대 이상"]:
    sub = sc[sc["age"] == a]
    for _, r in sub.iterrows():
        print(f"[{a}] cot_id={r['cot_id']} / 본문 연령 표현:", around(r["question"], r"[1-9]0\s*대|\d{2}\s*(?:세|살)", 25))

## C6. 범위 밖 CoT 분류용 표

- 한 행 = 범위 밖 추천이 있는 CoT 1건. 범위 밖 상품이 여러 개면 이어 붙인다.
- `data/review/범위밖_분류표.csv`로 저장한다 (원문 문장이 들어 있어 git 제외 폴더인 `data/` 아래에 둔다).
- `내 분류` 칸은 비워 둔다.

In [ ]:
out2 = a1[a1["허용범위"] == "범위 밖"]
rows = []
for cid, g in out2.groupby("cot_id"):
    r0 = sc[sc["cot_id"] == cid].iloc[0]
    gset, kind, raw = parse_allowed_grades(r0["cot2"], with_text=True)
    rows.append({
        "cot_id": cid,
        "질문유형": r0["query_group"],
        "성향_cot1": r0["성향_cot1"],
        "성향_cot2": r0["성향_cot2"],
        "허용등급(파싱)": "-".join(map(str, sorted(gset))) if gset else "",
        "허용등급 원문": " | ".join(raw),
        "범위 밖 추천 상품": " | ".join(g["rec_name"]),
        "추천 등급(JSON)": " | ".join(str(int(x)) for x in g["json_grade"]),
        "cot2 등급 문장": grade_sents(r0["cot2"], 1000),
        "내 분류": "",
    })
review = pd.DataFrame(rows)
out_dir = ROOT / "data" / "review"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "범위밖_분류표.csv"
# 이미 있으면 덮어쓰지 않는다 ('내 분류' 칸에 직접 적은 내용을 지우지 않기 위해). 새로 만들려면 파일을 지우고 다시 실행
if out_path.exists():
    print("이미 있어 덮어쓰지 않음:", out_path, "/ 지금 계산한 행 수:", len(review))
else:
    review.to_csv(out_path, index=False, encoding="utf-8-sig")   # 엑셀에서 한글이 깨지지 않게 utf-8-sig
    print("저장:", out_path, "/ 행 수:", len(review))
review.drop(columns=["cot2 등급 문장", "범위 밖 추천 상품"])

# 추가 확인 4 (2026-11-12)

판정 없이 사실만 확인한다.

| # | 항목 |
|---|---|
| D1 | cot2 성향 명칭과 허용 집합이 다른 성향의 다수 기준과 같은 경우 |
| D2 | cot2에서 숫자 없이 구분점으로 시작하는 등급 표기 ('허용·4·5·6등급' 등) |
| D3 | 추천 TDF 상품의 목표 연도 × 고객 연령대 |
| D4 | P 클래스(연금 클래스로 보이는 C-P, C-Pe 등) 추천 CoT의 질문 속 세제혜택 진술 |
| D5 | 상품 원천데이터 설명에 TDF 목표 연도·연금 클래스 설명이 있는지 |

## D1. cot2 성향 명칭 × 허용 집합

- 성향은 cot2에서 하나만 나온 경우(`성향_cot2`)만 쓴다. 없음·복수와 허용 집합 추출 실패는 뺀다.
- **다수 기준** = 그 성향에서 가장 많이 나온 허용 집합. 이 데이터 안의 최빈값일 뿐 규정상 기준이 아니다.

In [ ]:
d1 = sc[sc["성향_cot2"].isin(PROPENSITY) & (sc["허용등급"] != "추출 실패")]
majority = d1.groupby("성향_cot2")["허용등급"].agg(lambda s: s.value_counts().index[0])
print("성향별 다수 기준 (cot2 기준):")
print(majority)
print()
# 질문에서 요청한 경우: cot2에 '적극투자형'이라는 글자가 있고 허용 집합이 {4,5,6}
m_text = sc["cot2"].str.contains("적극투자형") & (sc["허용등급"] == "4-5-6")
ids = sorted(sc.loc[m_text, "cot_id"])
print("cot2에 '적극투자형' 글자 + 허용 {4,5,6}:", len(ids), ids)
print("718 포함:", 718 in ids, "/ 1313 포함:", 1313 in ids)
m_one = (sc["성향_cot2"] == "적극투자형") & (sc["허용등급"] == "4-5-6")
print("그중 cot2 성향이 적극투자형 하나뿐인 건:", m_one.sum(), sorted(sc.loc[m_one, "cot_id"]))

In [ ]:
# 성향별: 허용 집합이 '다른 성향의 다수 기준'과 같은 건수
# 애매한 경우 처리: 한 집합이 두 성향의 다수 기준과 같을 수는 없다(다수 기준 5개가 모두 다름) → 한 성향에만 대응
maj_to_p = {v: k for k, v in majority.items()}
d1 = d1.assign(같은_다수기준=d1["허용등급"].map(maj_to_p))
d1["구분"] = np.select([d1["같은_다수기준"] == d1["성향_cot2"], d1["같은_다수기준"].notna()],
                       ["자기 성향 다수 기준", "다른 성향 다수 기준"], "어느 다수 기준과도 다름")
print(pd.crosstab(d1["성향_cot2"], d1["구분"], margins=True))
print()
other = d1[d1["구분"] == "다른 성향 다수 기준"]
t = other.groupby(["성향_cot2", "같은_다수기준", "허용등급"])["cot_id"].agg(건수="size", cot_id=lambda s: sorted(s)[:15])
t

## D2. 숫자 없이 구분점으로 시작하는 등급 표기

- 한글 바로 뒤에 `·`(또는 `ㆍ`)가 오고 그 뒤에 등급 숫자가 오는 경우: 예) `허용·4·5·6등급`, `허용 위험등급은·4, 5, 6등급`
- 참고로 쉼표 바로 뒤 `·`가 오는 경우(`5,·6등급`)도 따로 센다.
- 각 건에서 `parse_allowed_grades`가 뽑은 허용 집합을 함께 보여 준다.

In [ ]:
PAT_HANGUL_DOT = r"[가-힣]\s*[·ㆍ]\s*[1-6]"
PAT_COMMA_DOT = r",\s*[·ㆍ]\s*[1-6]"
for name, pat in [("한글 뒤 구분점", PAT_HANGUL_DOT), ("쉼표 뒤 구분점", PAT_COMMA_DOT)]:
    hit = sc[sc["cot2"].str.contains(pat, regex=True)]
    print(f"[{name}] CoT {len(hit)}건")
    for _, r in hit.iterrows():
        mm = re.search(pat, r["cot2"])
        snip = r["cot2"][max(0, mm.start() - 30): mm.end() + 25].replace("\n", " ")
        print(f"  cot_id={r['cot_id']} | 파싱={r['허용등급']} | …{snip}…")
    print()

## D3. 추천 TDF 상품의 목표 연도 × 고객 연령대

- 추천 상품명에 'TDF'가 있는 경우. 연도는 상품명 안의 20xx 네 자리 숫자.
- 연령대는 메타 `age`. 상품명에 연도가 없으면 '연도 없음'.

In [ ]:
d3 = rec[rec["rec_name"].str.contains("TDF", case=False)].copy()
d3["TDF연도"] = d3["rec_name"].str.extract(r"(20[2-9]\d)")[0].fillna("연도 없음")
print("TDF 추천:", len(d3), "/ CoT:", d3["cot_id"].nunique(), "/ 질문유형:", d3["query_group"].value_counts().to_dict())
pd.crosstab(d3["age"], d3["TDF연도"], margins=True)

In [ ]:
# 연도 없는 TDF 상품명 (고유)
print(d3.loc[d3["TDF연도"] == "연도 없음", "rec_name"].value_counts().head(10))

## D4. P 클래스 추천 CoT의 질문 속 세제혜택 진술

- 클래스 = 상품명 끝 괄호 안 표기 (예: `(C-Pe)`). 하이픈 뒤에 P가 들어가면 P 클래스로 센다.
  대문자 P만 / 소문자 p 포함 두 가지로 센다. **P가 연금저축 클래스를 뜻하는지는 데이터에 정의가 없어 불확실하다.**
- 질문의 세제 진술: '세제'가 든 문장이 없으면 '언급 없음', 그 문장에 부정 표현(고려하지 않·필요 없·상관없·크게 신경 쓰지 않 등)이 있으면 '부정 진술'.
  애매한 경우 처리: 부정 표현 목록은 샘플을 보고 만든 것이라 빠진 표현이 있을 수 있다. '관계없이'도 부정 진술로 센다.

In [ ]:
NEG = (r"고려하지\s*않|필요(하지\s*않|\s*없|는\s*없)|중요하지\s*않|상관(없|하지\s*않)|없어도|관계없|무관|"
       r"신경\s*쓰지\s*않|우선순위가\s*아니|무방|원하지\s*않|관심(이|은)?\s*없|크게\s*(중요|신경)|필수\s*(사항)?(은|는|가)?\s*아니")

def tax_statement(q):
    sents = [s for s in re.split(r"(?<=[.!?])\s+", str(q)) if "세제" in s]
    if not sents:
        return "언급 없음"
    return "부정 진술" if any(re.search(NEG, s) for s in sents) else "언급 있음(부정 표현 없음)"

rec["클래스"] = rec["rec_name"].str.extract(r"\(([^()]*)\)\s*$")[0].fillna("")
rec["P클래스"] = rec["클래스"].str.contains(r"-[A-Za-z]*P", regex=True)
rec["P클래스(소문자 포함)"] = rec["클래스"].str.contains(r"-[A-Za-z]*[Pp]", regex=True)
qtax = cot.set_index("cot_id")["question"].map(tax_statement)

for col in ["P클래스", "P클래스(소문자 포함)"]:
    ids4 = rec.loc[rec[col], "cot_id"].unique()
    print(f"[{col}] 추천 상품 {rec[col].sum()}개 / CoT {len(ids4)}건")
    print(qtax.loc[ids4].value_counts().to_string())
    print()
print("참고: 증권 CoT 전체의 세제 진술 분포")
print(qtax.loc[sc["cot_id"]].value_counts().to_string())

In [ ]:
# P 클래스 추천 + 질문에 부정 진술: 건별 문장
ids_p = rec.loc[rec["P클래스(소문자 포함)"], "cot_id"].unique()
neg_ids = [c for c in ids_p if qtax[c] == "부정 진술"]
for c in neg_ids:
    q = cot.loc[cot["cot_id"] == c, "question"].iloc[0]
    s = [s for s in re.split(r"(?<=[.!?])\s+", q) if "세제" in s and re.search(NEG, s)][0]
    names = rec.loc[(rec["cot_id"] == c) & rec["P클래스(소문자 포함)"], "클래스"].tolist()
    print(f"cot_id={c} | P클래스={names} | {s[:120]}")

## D5. 상품 원천데이터 설명에 TDF 목표 연도·연금 클래스 설명이 있는지

- 대상: 상품 JSON(증권). 설명 문장은 `investment_strategy`에 있다.
- 키워드가 나온 상품 수와 샘플 3개씩 (키워드 앞뒤 일부만).

In [ ]:
ps = prod[prod["domain"] == "증권"].copy()
ps["클래스"] = ps["product_name"].str.extract(r"\(([^()]*)\)\s*$")[0].fillna("")
ps["P클래스"] = ps["클래스"].str.contains(r"-[A-Za-z]*[Pp]", regex=True)
ps_tdf = ps[ps["product_name"].str.contains("TDF", na=False)].copy()
strat = ps["investment_strategy"].fillna("")

# TDF: 설명 속 목표시점 연도가 상품명 연도와 같은지
ps_tdf["이름연도"] = ps_tdf["product_name"].str.extract(r"(20[2-9]\d)")[0]
ps_tdf["설명연도"] = strat[ps_tdf.index].str.extract(r"목표\s*시점[^.]{0,40}?(20[2-9]\d)")[0]
print("TDF 상품 JSON:", len(ps_tdf))
print("설명에 '목표시점 … 20xx' 있음:", ps_tdf["설명연도"].notna().sum())
both = ps_tdf.dropna(subset=["이름연도", "설명연도"])
print("이름·설명 연도 둘 다 있음:", len(both), "/ 일치:", (both["이름연도"] == both["설명연도"]).sum())
print()
for kw in ["연금저축", "개인연금", "퇴직연금", "클래스", "종류"]:
    print(f"'{kw}' 포함 — P클래스 상품 {strat[ps['P클래스']].str.contains(kw).sum()}/{ps['P클래스'].sum()}, "
          f"그 외 {strat[~ps['P클래스']].str.contains(kw).sum()}/{(~ps['P클래스']).sum()}")
print()
print("tax_type × P클래스")
print(pd.crosstab(ps["P클래스"], ps["tax_type"], margins=True))

In [ ]:
def around(t, kw, before=60, after=140):
    i = t.find(kw)
    return t[max(0, i - before): i + after].replace("\n", " ")

print("== TDF 목표시점 샘플 3")
for _, r in ps_tdf[ps_tdf["설명연도"].notna()].sample(3, random_state=0).iterrows():
    print(f"- {r['product_name']}\n   …{around(r['investment_strategy'], '목표')}…")
print()
print("== P클래스 상품 중 '연금' 언급 샘플 3")
pk = ps[ps["P클래스"] & strat.str.contains("연금저축|개인연금")]
pk = pk if len(pk) >= 3 else ps[ps["P클래스"] & strat.str.contains("연금")]
for _, r in pk.sample(min(3, len(pk)), random_state=0).iterrows():
    kw = "연금저축" if "연금저축" in r["investment_strategy"] else ("개인연금" if "개인연금" in r["investment_strategy"] else "연금")
    print(f"- {r['product_name']} (tax_type={r['tax_type']})\n   …{around(r['investment_strategy'], kw)}…")

# 추가 확인 5 (2026-11-12)

판정 없이 사실만 확인한다.

| # | 항목 |
|---|---|
| E1 | D2의 1190, 2074, 4123의 cot2 성향 |
| E2 | 안정추구형 {4,5,6} / 위험중립형 3등급 포함 범위에서 그 등급을 실제 추천한 건수 |
| E3 | 적극투자형 '어느 다수 기준과도 다름' 67건의 허용 집합 종류 |
| E4 | 상품명과 설명의 목표 연도가 다른 TDF 상품 목록 |

## E1. D2 사례의 cot2 성향

In [ ]:
e1 = sc[sc["cot_id"].isin([1190, 2074, 4123, 718])][["cot_id", "split", "query_group", "성향_cot1", "성향_cot2", "허용등급"]]
e1.sort_values("cot_id")

## E2. 넓은 허용 범위에서 그 등급을 실제 추천했는지

- 추천 등급은 상품 JSON `risk_grade` 기준. JSON 등급이 없는 추천(매칭 실패·등급 복수)은 세지 않는다.
- 안정추구형: cot2 성향 = 안정추구형, 허용 집합 = {4,5,6}, 그중 **4등급** 상품을 추천한 경우.
- 위험중립형: D1에서 '어느 다수 기준과도 다름'(705건)의 허용 집합 분포를 먼저 보이고, 그중 **3이 포함된 집합**에서 **3등급** 상품을 추천한 경우를 센다.

In [ ]:
def e2_count(cots, grade, label):
    # 대상 CoT 안에서 해당 등급 추천 수(상품 단위)와 CoT 수를 split × 질문유형별로 센다
    r = rec[rec["cot_id"].isin(cots["cot_id"]) & (rec["json_grade"] == grade)]
    base = cots.groupby(["split", "query_group"]).size().rename("대상 CoT")
    hit_cot = r.groupby(["split", "query_group"])["cot_id"].nunique().rename(f"{grade}등급 추천 CoT")
    hit_n = r.groupby(["split", "query_group"]).size().rename(f"{grade}등급 추천 수")
    t = pd.concat([base, hit_cot, hit_n], axis=1).fillna(0).astype(int)
    t.loc[("합계", ""), :] = t.sum()
    print(f"[{label}]")
    print(t)
    print()
    return r

ast = d1[(d1["성향_cot2"] == "안정추구형") & (d1["허용등급"] == "4-5-6")]
r_ast = e2_count(ast, 4, "안정추구형 + {4,5,6}")

neu_other = d1[(d1["성향_cot2"] == "위험중립형") & (d1["구분"] == "어느 다수 기준과도 다름")]
print("위험중립형 '어느 다수 기준과도 다름' 허용 집합 분포:")
print(neu_other["허용등급"].value_counts().to_string())
print()
neu3 = neu_other[neu_other["허용등급"].str.split("-").map(lambda g: "3" in g)]
print("그중 3 포함 집합:", len(neu3))
r_neu = e2_count(neu3, 3, "위험중립형 + 3 포함 집합")

## E3. 적극투자형 '어느 다수 기준과도 다름' 67건의 허용 집합

In [ ]:
e3 = d1[(d1["성향_cot2"] == "적극투자형") & (d1["구분"] == "어느 다수 기준과도 다름")]
print("건수:", len(e3))
e3.groupby("허용등급")["cot_id"].agg(건수="size", cot_id=lambda s: sorted(s)[:12]).sort_values("건수", ascending=False)

## E4. 상품명과 설명의 목표 연도가 다른 TDF 상품 (D5)

- 설명 연도는 `investment_strategy`에서 '목표시점' 뒤 40자 안의 첫 20xx. 추출 방식 때문에 다르게 잡혔을 가능성도 있어 해당 부분 원문을 함께 보인다.

In [ ]:
e4 = ps_tdf.dropna(subset=["이름연도", "설명연도"])
e4 = e4[e4["이름연도"] != e4["설명연도"]].copy()
e4["설명 원문"] = e4["investment_strategy"].map(lambda t: around(t, "목표", 20, 90))
e4["CoT 추천 횟수"] = e4["product_name"].map(rec["rec_name"].value_counts()).fillna(0).astype(int)
print("개수:", len(e4))
e4[["product_name", "이름연도", "설명연도", "CoT 추천 횟수", "설명 원문"]].reset_index(drop=True)

# 추가 확인 6 (2026-11-12)

판정 없이 사실만 확인한다. (D37 가설 확인용)

| # | 항목 |
|---|---|
| F1 | 증권_02: question의 고객 자기 진술 성향 vs cot1·cot2 성향 |
| F2 | question에 '3등급'·'다소 높은 위험' 언급 여부별, 공개 기준 범위 밖 추천 비율 (성향별) |
| F3 | F2 언급 CoT 중 '범위 초과' 명시·상담/확인 안내 사례 |
| F4 | 교정 후보 자동 추출: 성향-범위 짝은 공개 기준과 다르지만 추천은 공개 기준 안 |
| F5 | cot 6799 추천 상품의 원천데이터 전체 필드 |

**공개 기준 범위 (이 노트북에서 쓰는 값)**
- 위험중립형 4~6, 안정추구형 5~6: D30 근거(docs/sources.md 4·5번).
- 공격투자형 1~6.
- 적극투자형 2~6, 안정형 6: D43 (한국투자증권 투자권유준칙 제6조①). 적극투자형 3~6(은행 기준)은 더 보수적이라 결함으로 보지 않는다(D31).

**성향 (D21)**: cot2에 성향이 하나만 있으면 그것을 쓰고, 아니면 cot1에 하나만 있을 때 그것을 쓴다. 둘 다 아니면 '비교 불가'.

In [ ]:
ORDER = {p: i for i, p in enumerate(["안정형", "안정추구형", "위험중립형", "적극투자형", "공격투자형"])}
# 공개 기준 (D43): 공격 1~6, 적극 2~6, 위험중립 4~6, 안정추구 5~6, 안정 6
PUBLIC = {"공격투자형": {1, 2, 3, 4, 5, 6}, "적극투자형": {2, 3, 4, 5, 6}, "위험중립형": {4, 5, 6},
          "안정추구형": {5, 6}, "안정형": {6}}
PUBLIC_V = {"공개기준": PUBLIC}

# D21: cot2 단일 성향 우선, 없으면 cot1 단일 성향
sc["성향_D21"] = np.where(sc["성향_cot2"].isin(PROPENSITY), sc["성향_cot2"],
                        np.where(sc["성향_cot1"].isin(PROPENSITY), sc["성향_cot1"], "비교 불가"))
sc["성향_질문"] = sc["question"].map(propensity_in)
print(sc["성향_D21"].value_counts().to_string())

## F1. 증권_02: 고객 자기 진술 성향 vs cot1·cot2 성향

- question에서 성향 명칭이 하나만 나오면 그것을 고객 자기 진술 성향으로 본다.
  애매한 경우 처리: "안정형보다는 …"처럼 자기 성향이 아닌 맥락의 언급도 잡힐 수 있다. 없음·복수는 '비교 불가'.
- 상향 = CoT 쪽 성향이 더 공격적, 하향 = 더 보수적.

In [ ]:
def direction(q, c):
    if q not in ORDER or c not in ORDER:
        return "비교 불가"
    return "같음" if ORDER[q] == ORDER[c] else ("상향" if ORDER[c] > ORDER[q] else "하향")

f1 = sc[sc["query_group"] == "증권_02.투자성향 및 조건기반형"].copy()
print("증권_02 CoT:", len(f1), "/ 질문 성향:", f1["성향_질문"].value_counts().to_dict())
print()
for col in ["성향_cot1", "성향_cot2", "성향_D21"]:
    f1[f"방향_{col}"] = [direction(q, c) for q, c in zip(f1["성향_질문"], f1[col])]
tbl = pd.DataFrame({col: f1[f"방향_{col}"].value_counts() for col in ["성향_cot1", "성향_cot2", "성향_D21"]}).fillna(0).astype(int)
tbl.loc[["같음", "상향", "하향", "비교 불가"]]

In [ ]:
# 상향·하향 쌍 분포와 샘플 cot_id 3개씩 (D21 성향 기준)
for d in ["상향", "하향"]:
    sub = f1[f1["방향_성향_D21"] == d]
    print(f"[{d}] {len(sub)}건")
    print(sub.groupby(["성향_질문", "성향_D21"]).size().to_string())
    print("샘플 cot_id:", sorted(sub["cot_id"].sample(min(3, len(sub)), random_state=0)))
    print()

## F2. question에 '3등급'·'다소 높은 위험' 언급 여부별 공개 기준 범위 밖 추천 비율

- 언급 = question에 `3등급`(공백 허용) 또는 `다소 높은 위험`(공백 무관)이 있음.
- 범위 밖 추천 CoT = 추천 상품 중 JSON 등급이 그 성향의 공개 기준 범위 밖인 것이 하나라도 있는 CoT.
- 대상: D21 성향이 있는 증권 CoT 중 JSON 등급이 있는 추천이 1개 이상인 CoT.

In [ ]:
MENTION = r"3\s*등급|다소\s*높은\s*위험"
sc["3등급 언급"] = sc["question"].str.contains(MENTION, regex=True)
f2r = rec[(rec["category"] == "증권") & rec["json_grade"].notna()].merge(
    sc[["cot_id", "성향_D21", "3등급 언급"]], on="cot_id")
f2r = f2r[f2r["성향_D21"] != "비교 불가"]
print("question 언급 CoT (증권 전체):", sc["3등급 언급"].sum(), "/", len(sc))
for v, pub in PUBLIC_V.items():
    f2r[f"밖_{v}"] = [g not in pub[p] for g, p in zip(f2r["json_grade"], f2r["성향_D21"])]
f2 = f2r.groupby(["cot_id", "성향_D21", "3등급 언급"])[[f"밖_{v}" for v in PUBLIC_V]].any().reset_index()

def rate(df, col):
    t = df.groupby(["성향_D21", "3등급 언급"])[col].agg(CoT="size", 범위밖="sum")
    t["비율(%)"] = (t["범위밖"] / t["CoT"] * 100).round(1)
    return t.unstack("3등급 언급")

for v in PUBLIC_V:
    print(f"[공개 기준: {v}]")
    print(rate(f2, f"밖_{v}"))
    print()

## F3. 언급 CoT 중 '범위 초과' 명시 또는 상담·확인 안내

- 대상: F2에서 question에 언급이 있는 증권 CoT.
- 찾는 곳: cot1·cot2·cot3·answer. 키워드 기준이라 문맥(예: 상품 일반 설명 속 '상담')은 가리지 않는다. 샘플로 확인한다.

In [ ]:
EXCEED = r"범위를?\s*(?:초과|벗어|넘)|허용\s*범위\s*(?:밖|외)|초과하(?:는|여|며|므로)|범위\s*초과"
GUIDE = r"상담(?:사|원|을|이|\s*창구)|전문가|재진단|(?:투자)?성향\s*(?:을\s*)?(?:다시|재)\s*(?:진단|평가|확인|분석)|투자자\s*정보\s*확인|부적합\s*(?:확인|안내|금융)|확인서"
f3 = sc[sc["3등급 언급"]].copy()
body = f3["cot1"] + " " + f3["cot2"] + " " + f3["cot3"] + " " + f3["answer"]
f3["범위 초과 명시"] = body.str.contains(EXCEED, regex=True)
f3["상담·확인 안내"] = body.str.contains(GUIDE, regex=True)
print("대상:", len(f3))
print(pd.crosstab(f3["범위 초과 명시"], f3["상담·확인 안내"], margins=True))
# 범위 밖 추천(적극 3~6 기준) CoT와 겹치는 정도
out_ids = set(f2.loc[f2["밖_공개기준"], "cot_id"])
f3["범위 밖 추천"] = f3["cot_id"].isin(out_ids)
print()
print(pd.crosstab(f3["범위 밖 추천"], [f3["범위 초과 명시"], f3["상담·확인 안내"]]))

In [ ]:
# 샘플: 키워드 앞뒤 일부
def kw_snip(text, pat, w=70):
    m = re.search(pat, text)
    return text[max(0, m.start() - w): m.end() + w].replace("\n", " ") if m else ""

for name, pat in [("범위 초과 명시", EXCEED), ("상담·확인 안내", GUIDE)]:
    sub = f3[f3[name]]
    print(f"== {name} 샘플 (최대 5)")
    for _, r in sub.sample(min(5, len(sub)), random_state=0).iterrows():
        txt = r["cot1"] + " " + r["cot2"] + " " + r["cot3"] + " " + r["answer"]
        print(f"  cot_id={r['cot_id']} | {r['성향_D21']} | …{kw_snip(txt, pat)}…")
    print()

## F4. 교정 후보 자동 추출

- 조건: cot2 성향(하나만 나온 경우)과 cot2 허용 집합의 짝이 공개 기준 짝과 다름 **그리고** JSON 등급이 있는 추천이 모두 그 성향의 공개 기준 범위 안.
- JSON 등급이 없는 추천이 섞인 CoT는 뺀다 (모두 범위 안인지 알 수 없음).
- 목록은 `data/review/교정후보_자동추출.csv`로 저장한다 (원문 문장 없음).

In [ ]:
recg = rec[rec["category"] == "증권"].groupby("cot_id")["json_grade"].agg(list)
f4 = sc[sc["성향_cot2"].isin(PROPENSITY) & sc["허용집합"].notna()][["cot_id", "split", "query_group", "성향_cot2", "허용등급"]].copy()
f4["추천 등급"] = f4["cot_id"].map(recg)
f4 = f4[f4["추천 등급"].map(lambda g: isinstance(g, list) and len(g) > 0 and all(pd.notna(x) for x in g))]
for v, pub in PUBLIC_V.items():
    # 애매한 경우 처리: 적극투자형 3~6은 더 보수적인 은행 기준이라 짝 불일치로 보지 않는다 (D31)
    pair_diff = [set(map(int, s.split("-"))) != pub[p] and not (p == "적극투자형" and s == "3-4-5-6")
                 for s, p in zip(f4["허용등급"], f4["성향_cot2"])]
    inside = [all(int(x) in pub[p] for x in g) for g, p in zip(f4["추천 등급"], f4["성향_cot2"])]
    f4[f"후보_{v}"] = np.array(pair_diff) & np.array(inside)
for v in PUBLIC_V:
    sub = f4[f4[f"후보_{v}"]]
    print(f"[{v}] 후보 CoT {len(sub)}건")
    print(sub.groupby(["성향_cot2", "허용등급"]).size().to_string())
    print()
cand = f4[f4[[f"후보_{v}" for v in PUBLIC_V]].any(axis=1)].copy()
cand["추천 등급"] = cand["추천 등급"].map(lambda g: "|".join(str(int(x)) for x in g))
cand = cand.rename(columns={"성향_cot2": "성향", "허용등급": "적힌 범위"})
cand.to_csv(ROOT / "data" / "review" / "교정후보_자동추출.csv", index=False, encoding="utf-8-sig")
print("저장 행 수:", len(cand))
cand.head(10)

## F5. cot 6799 추천 상품의 원천데이터 전체 필드

In [ ]:
r6799 = rec[rec["cot_id"] == 6799]
print(r6799[["rec_name", "match_stage", "json_grade"]].to_string())
name = "KB스타 막강 단기 국공채 증권 모투자신탁(채권)(신탁)"
keys = match_product(name, idx)[1]
print("매칭 단계:", match_product(name, idx)[0], "/ 키:", keys)
for k in keys:
    row = idx["table"].loc[k]
    for c, v in row.items():
        if pd.notna(v) and str(v).strip() != "":
            print(f"- {c}: {v}")

## F6. D30 퇴직연금 태그 대상 CoT의 cot1~answer 표현 (D30 근거 확인)

- 대상: D30 넓은 정의 — 완화된 등급 추천(안정추구형 {4,5,6}의 4등급, 위험중립형 3~6의 3등급) 중
  상품명·원천 설명·원천 상품 유형 어디든 '퇴직연금'이 있는 추천을 가진 CoT.
- 표현별로 cot1·cot2·cot3·answer 각각에 나오는 CoT 수를 센다.
- '허용 범위를 넓히거나 예외를 두는 이유' 후보 문장: 한 문장 안에 (퇴직연금·완화·예외·장기·적립) 중 하나와
  (허용·범위·등급 수용·감수) 중 하나가 함께 있는 문장. 키워드 기준이므로 샘플로 직접 확인한다.

In [ ]:
tbl_p = idx["table"]
def _has_pension(keys):
    return any(("퇴직연금" in str(tbl_p.loc[k, "investment_strategy"])) or ("퇴직연금" in str(tbl_p.loc[k, "product_type"]))
               for k in keys if k in tbl_p.index)

wide_r = rec[rec["category"] == "증권"].merge(sc[["cot_id", "성향_cot2", "허용등급"]], on="cot_id")
wide_r = wide_r[((wide_r["성향_cot2"] == "안정추구형") & (wide_r["허용등급"] == "4-5-6") & (wide_r["json_grade"] == 4)) |
                ((wide_r["성향_cot2"] == "위험중립형") & (wide_r["허용등급"] == "3-4-5-6") & (wide_r["json_grade"] == 3))]
wide_r = wide_r[wide_r["rec_name"].str.contains("퇴직연금") | wide_r["keys"].map(_has_pension)]
f6 = sc[sc["cot_id"].isin(wide_r["cot_id"])].copy()
print("대상 CoT:", len(f6), f6["성향_cot2"].value_counts().to_dict())

TERMS = {"퇴직연금": r"퇴직연금", "완화": r"완화", "적립": r"적립", "위험 회피": r"위험\s*회피|위험을\s*회피",
         "장기 투자": r"장기\s*투자", "예외": r"예외"}
parts = ["cot1", "cot2", "cot3", "answer"]
t6 = pd.DataFrame({t: [f6[p].str.contains(pat, regex=True).sum() for p in parts] +
                      [f6[parts].apply(lambda r: any(re.search(pat, str(x)) for x in r), axis=1).sum()]
                   for t, pat in TERMS.items()}, index=parts + ["어디든"])
t6

In [ ]:
REASON_A = r"퇴직연금|완화|예외|장기|적립"
REASON_B = r"허용|범위|수용|감수"
hits = []
for _, r in f6.iterrows():
    for p in parts:
        for s in re.split(r"(?<=[.!?])\s+", str(r[p])):
            if re.search(REASON_A, s) and re.search(REASON_B, s):
                hits.append((r["cot_id"], r["성향_cot2"], p, s.strip()))
h6 = pd.DataFrame(hits, columns=["cot_id", "성향", "위치", "문장"])
print("후보 문장이 있는 CoT:", h6["cot_id"].nunique(), "/ 문장 수:", len(h6))
print(h6["위치"].value_counts().to_string())
print()
for cid in h6["cot_id"].drop_duplicates().sample(min(5, h6["cot_id"].nunique()), random_state=0):
    for _, r in h6[h6["cot_id"] == cid].head(2).iterrows():
        print(f"cot_id={r['cot_id']} ({r['성향']}, {r['위치']}) {r['문장'][:200]}")

# 추가 확인 7 (2026-11-12)

판정 없이 사실만 확인한다. 공개 기준은 D43(공격 1~6, 적극 2~6, 위험중립 4~6, 안정추구 5~6, 안정 6).

| # | 항목 |
|---|---|
| G1 | 원천데이터: 상품명 핵심어가 investment_strategy에 하나도 없는 상품 (다른 상품 설명 의심) |
| G2 | D30 태그 674건 중 성향이 일관된 건 → 범위 표기 교정 후보, 무작위 30건 검수표 |

## G1. 상품명 핵심어가 설명에 하나도 없는 상품 (증권)

- 핵심어 목록(아래 `NAME_KW`) 중 상품명에 나온 것을 그 상품의 핵심어로 본다.
- 핵심어가 하나 이상 있는데 `investment_strategy`에 그중 하나도 없으면 '불일치 의심'.
- 애매한 경우 처리: 핵심어가 하나도 없는 상품은 판단할 수 없어 따로 센다. 동의어(예: 국공채↔국채)는 같은 묶음으로 본다.
  설명이 짧거나 일반적인 문구뿐이어도 불일치로 잡힐 수 있어 샘플로 확인한다.

In [ ]:
# 묶음: 상품명에 왼쪽 표현이 있으면, 설명에 오른쪽 표현 중 하나라도 있으면 일치로 본다
NAME_KW = {
    "국공채": ["국공채", "국채", "공채", "국고채"], "단기": ["단기"], "채권": ["채권"], "주식": ["주식"],
    "TDF": ["TDF", "Target Date", "목표시점", "타겟데이트"], "리츠": ["리츠", "REITs", "REIT", "부동산"],
    "부동산": ["부동산", "리츠", "REITs"], "배당": ["배당"], "인덱스": ["인덱스", "지수"],
    "미국": ["미국", "US", "북미"], "유럽": ["유럽"], "중국": ["중국", "차이나"], "일본": ["일본"],
    "베트남": ["베트남"], "인도": ["인도"], "아시아": ["아시아"], "브라질": ["브라질"],
    "이머징": ["이머징", "신흥"], "신흥": ["신흥", "이머징"], "글로벌": ["글로벌", "해외", "전세계", "전 세계", "외국"],
    "골드": ["금", "골드"], "원자재": ["원자재", "커머디티"], "반도체": ["반도체"], "헬스케어": ["헬스케어", "바이오"],
    "달러": ["달러", "USD", "미국"], "하이일드": ["하이일드", "투기등급"], "MMF": ["MMF", "단기금융"],
}
ps = prod[prod["domain"] == "증권"].copy()
ps["핵심어"] = ps["product_name"].map(lambda n: [k for k in NAME_KW if k in str(n)])
def kw_found(row):
    d = str(row["investment_strategy"])
    return [k for k in row["핵심어"] if any(v in d for v in NAME_KW[k])]
ps["설명에 있는 핵심어"] = ps.apply(kw_found, axis=1)
ps["G1"] = np.where(ps["핵심어"].str.len() == 0, "핵심어 없음(판단 불가)",
                    np.where(ps["설명에 있는 핵심어"].str.len() == 0, "불일치 의심", "하나 이상 일치"))
print(ps["G1"].value_counts().to_string())
print()
g1 = ps[ps["G1"] == "불일치 의심"]
print(g1["product_group"].value_counts().to_string())
print()
print("6799 상품 포함:", (g1["product_name"] == "KB스타 막강 단기 국공채 증권 모투자신탁(채권)(신탁)").any())
g1_rec = rec[rec["rec_name"].isin(g1["product_name"])]
print("불일치 의심 상품을 추천한 CoT:", g1_rec["cot_id"].nunique(), "/ 추천:", len(g1_rec))

In [ ]:
# 샘플 5개: 상품명, 핵심어, 설명 앞 150자
for _, r in g1.sample(min(5, len(g1)), random_state=0).iterrows():
    print(f"- {r['product_name']} | 등급 {r['risk_grade']} | 핵심어 {r['핵심어']}")
    print("   설명:", str(r["investment_strategy"])[:150].replace("\n", " "))

## G2. D30 태그 674건 중 성향이 일관된 건 (범위 표기 교정 후보)

- D30 태그 대상 = 안정추구형 + {4,5,6}이면서 추천이 모두 5~6등급, 위험중립형 + 3~6이면서 추천이 모두 4~6등급 (JSON 등급 기준).
- 조건: cot1 성향 = cot2 성향 (둘 다 하나만 나온 경우). 증권_02는 question의 고객 자기 진술 성향도 같아야 한다.
  증권_01·03은 question 성향 조건을 두지 않는다.
- 교정 범위 = 그 성향의 공개 기준(D43). **교정은 적용하지 않는다.**
- 무작위 30건을 `data/review/교정검수표.csv`로 저장한다 (원문 문장이 들어 있어 git 제외 폴더).

In [ ]:
grades_by_cot = rec[rec["category"] == "증권"].groupby("cot_id")["json_grade"].agg(list)
g2 = sc[((sc["성향_cot2"] == "안정추구형") & (sc["허용등급"] == "4-5-6")) |
        ((sc["성향_cot2"] == "위험중립형") & (sc["허용등급"] == "3-4-5-6"))].copy()
g2["추천 등급 목록"] = g2["cot_id"].map(grades_by_cot)
g2 = g2[g2["추천 등급 목록"].map(lambda g: isinstance(g, list) and len(g) > 0 and all(pd.notna(x) for x in g))]
g2 = g2[[all(int(x) in PUBLIC[p] for x in g) for g, p in zip(g2["추천 등급 목록"], g2["성향_cot2"])]]
print("D30 태그 대상:", len(g2), g2["성향_cot2"].value_counts().to_dict())

g2["cot1=cot2"] = g2["성향_cot1"] == g2["성향_cot2"]
is02 = g2["query_group"] == "증권_02.투자성향 및 조건기반형"
g2["질문=cot2"] = g2["성향_질문"] == g2["성향_cot2"]
g2["교정 후보"] = g2["cot1=cot2"] & (~is02 | g2["질문=cot2"])
print()
print(pd.crosstab([g2["성향_cot2"], g2["query_group"]], g2["교정 후보"], margins=True))
print()
print("cot1 성향 분포 (cot1≠cot2인 건):", g2.loc[~g2["cot1=cot2"], "성향_cot1"].value_counts().to_dict())
print("증권_02 중 질문 성향 ≠ cot2:", (is02 & g2["cot1=cot2"] & ~g2["질문=cot2"]).sum())

In [ ]:
def range_sentence(t):
    # cot2에서 허용 등급 근거 표현이 들어 있는 첫 문장
    _, _, texts = parse_allowed_grades(t, with_text=True)
    for s in re.split(r"(?<=[.!?])\s+", str(t)):
        if texts and texts[0] in s.replace("～", "~"):
            return s.strip()
    return texts[0] if texts else ""

cand2 = g2[g2["교정 후보"]].sample(30, random_state=0)
review2 = pd.DataFrame({
    "cot_id": cand2["cot_id"],
    "성향": cand2["성향_cot2"],
    "적힌 범위": cand2["허용등급"],
    "교정 범위": cand2["성향_cot2"].map(lambda p: "-".join(map(str, sorted(PUBLIC[p])))),
    "추천 등급": cand2["추천 등급 목록"].map(lambda g: "|".join(str(int(x)) for x in g)),
    "cot2 범위 문장": cand2["cot2"].map(range_sentence),
    "내 판정": "",
}).sort_values("cot_id")
out_path2 = ROOT / "data" / "review" / "교정검수표.csv"
if out_path2.exists():
    print("이미 있어 덮어쓰지 않음:", out_path2)
else:
    review2.to_csv(out_path2, index=False, encoding="utf-8-sig")
    print("저장:", out_path2, "/ 행 수:", len(review2))
print("문장 못 찾은 행:", (review2["cot2 범위 문장"] == "").sum())
review2.drop(columns=["cot2 범위 문장"]).groupby(["성향", "적힌 범위"]).size()

# 추가 확인 8 (2026-11-13)

판정 없이 사실만 확인한다.

| # | 항목 |
|---|---|
| H1 | G1 재집계: 클래스·유형 표시를 뺀 상품명 핵심어로 불일치 의심 상품 수 |
| H2 | investment_strategy 본문의 'N등급'과 위험등급 필드가 다른 상품, 추천 CoT, D30·D42 집계 포함 여부 |
| H3 | H1 재집계 후 의심 상품 무작위 20개 → `data/review/원천검수표.csv` |
| H4 | 교정검수표에 'cot2 안에 허용 범위·넓은 등급을 언급하는 문장 수' 열 추가 |

## H1. G1 재집계 (클래스·유형 표시 제외)

- G1의 핵심어 목록(`NAME_KW`)에는 퇴직연금·연금·온라인·클래스 기호가 원래 없다. 그래서 '목록에서 그 단어를 뺀다'만으로는 결과가 G1과 같다.
- 클래스·유형 표시는 상품명의 괄호 안에 있다. 예) `(C-Pe)`, `(수수료미징구-온라인-퇴직연금(C))`, `[주식]`, `(채권혼합)`.
  그래서 **괄호 `( )`·`[ ]` 안을 모두 지운 상품명**에서 핵심어를 다시 찾는다.
- 애매한 경우 처리: `[주식]`, `(채권)` 같은 유형 표시도 함께 빠진다. 핵심어가 괄호 밖에 없으면 '판단 불가'로 간다.

In [ ]:
def strip_brackets(name):
    # 중첩 괄호까지 지우기 위해 바깥쪽부터 반복
    s, prev = str(name), None
    while s != prev:
        prev = s
        s = re.sub(r"\([^()]*\)|\[[^\[\]]*\]", " ", s)
    return s

ps["이름_괄호제외"] = ps["product_name"].map(strip_brackets)
ps["핵심어2"] = ps["이름_괄호제외"].map(lambda n: [k for k in NAME_KW if k in n])
ps["설명에 있는 핵심어2"] = ps.apply(lambda r: [k for k in r["핵심어2"]
                                          if any(v in str(r["investment_strategy"]) for v in NAME_KW[k])], axis=1)
ps["H1"] = np.where(ps["핵심어2"].str.len() == 0, "핵심어 없음(판단 불가)",
                    np.where(ps["설명에 있는 핵심어2"].str.len() == 0, "불일치 의심", "하나 이상 일치"))
print(pd.crosstab(ps["G1"], ps["H1"], margins=True))
print()
h1 = ps[ps["H1"] == "불일치 의심"]
print("H1 불일치 의심:", len(h1), "/ 상품 그룹:", h1["product_group"].value_counts().to_dict())
print("6799 상품 포함:", (h1["product_name"] == "KB스타 막강 단기 국공채 증권 모투자신탁(채권)(신탁)").any())
h1_rec = rec[rec["rec_name"].isin(h1["product_name"])]
print("H1 의심 상품을 추천한 CoT:", h1_rec["cot_id"].nunique(), "/ 추천:", len(h1_rec))

## H2. 본문의 'N등급'과 위험등급 필드가 다른 상품

- 본문 등급: `investment_strategy`에서 'N등급으로 분류' 또는 'N 등급으로 분류'의 N (운용사가 등급을 밝히는 정형 문장).
  애매한 경우 처리: 이 문장이 없으면 '본문 등급 없음'. 여러 번 나오고 값이 다르면 '본문 등급 복수'로 두고 비교하지 않는다.
- 필드 등급: `risk_grade`의 숫자.
- D30·D42 포함 여부는 상품명 기준으로 그 상품이 해당 집계의 추천에 들어 있는지 본다.

In [ ]:
from src.extract import grade_num

def body_grade(t):
    gs = set(re.findall(r"([1-6])\s*등급\s*으로\s*분류", str(t)))
    if not gs:
        return "본문 등급 없음"
    return int(gs.pop()) if len(gs) == 1 else "본문 등급 복수"

ps["본문 등급"] = ps["investment_strategy"].map(body_grade)
ps["필드 등급"] = ps["risk_grade"].map(grade_num)
has_body = ps["본문 등급"].map(lambda x: isinstance(x, int))
ps["H2"] = np.where(~has_body, ps["본문 등급"].astype(str),
                    np.where(ps["본문 등급"].where(has_body) == ps["필드 등급"], "같음", "다름"))
print(ps["H2"].value_counts().to_string())
h2 = ps[ps["H2"] == "다름"].copy()
h2["추천 CoT 수"] = h2["product_name"].map(rec.groupby("rec_name")["cot_id"].nunique()).fillna(0).astype(int)
print()
print("등급 다른 상품:", len(h2), "/ 추천된 상품:", (h2["추천 CoT 수"] > 0).sum(),
      "/ 추천 CoT:", rec.loc[rec["rec_name"].isin(h2["product_name"]), "cot_id"].nunique())
print(pd.crosstab(h2["필드 등급"], h2["본문 등급"].astype(int), margins=True))

In [ ]:
# 목록 (상품명, 본문 등급, 필드 등급, 추천 CoT 수)
h2[["product_name", "본문 등급", "필드 등급", "추천 CoT 수"]].sort_values("추천 CoT 수", ascending=False).reset_index(drop=True)

In [ ]:
# D30·D42 집계에 이 상품들이 들어 있는지
mis = set(h2["product_name"])
# D30 제외 308건: 완화 등급(안정추구형 4등급, 위험중립형 3등급)을 추천한 CoT
d30w = rec[rec["category"] == "증권"].merge(sc[["cot_id", "성향_cot2", "허용등급"]], on="cot_id")
d30_ex = d30w[((d30w["성향_cot2"] == "안정추구형") & (d30w["허용등급"] == "4-5-6") & (d30w["json_grade"] == 4)) |
              ((d30w["성향_cot2"] == "위험중립형") & (d30w["허용등급"] == "3-4-5-6") & (d30w["json_grade"] == 3))]
x = d30_ex[d30_ex["rec_name"].isin(mis)]
print(f"D30 제외(완화 등급 추천 {d30_ex['cot_id'].nunique()} CoT / {len(d30_ex)} 추천) 중 등급 불일치 상품 추천:",
      x["cot_id"].nunique(), "CoT /", len(x), "추천")
# 이 중 본문 등급이 공개 기준 안이면 '완화 등급 추천'이 아니게 되는 건
x = x.merge(h2[["product_name", "본문 등급"]], left_on="rec_name", right_on="product_name")
x["본문 등급이 공개 기준 안"] = [int(b) in PUBLIC[p] for b, p in zip(x["본문 등급"], x["성향_cot2"])]
print("  그중 본문 등급으로 보면 공개 기준 안인 추천:", x["본문 등급이 공개 기준 안"].sum())
# 본문 등급을 쓰면 완화 등급 추천이 하나도 남지 않는 CoT (= 제외 대상에서 빠질 수 있는 CoT)
ok_names = set(x.loc[x["본문 등급이 공개 기준 안"], "rec_name"])
still = d30_ex[~d30_ex["rec_name"].isin(ok_names)]
print("  본문 등급을 쓰면 완화 등급 추천이 남지 않는 D30 제외 CoT:",
      d30_ex["cot_id"].nunique() - still["cot_id"].nunique())
# D30 태그 674건
x2 = rec[rec["cot_id"].isin(g2["cot_id"]) & rec["rec_name"].isin(mis)]
print("D30 태그 674 중 등급 불일치 상품 추천:", x2["cot_id"].nunique(), "CoT /", len(x2), "추천")
# D42: 위험중립형 3등급 언급 128건(공개 기준 밖 추천)
d42 = f2r[(f2r["성향_D21"] == "위험중립형") & f2r["3등급 언급"] & f2r["밖_공개기준"]]
x3 = d42[d42["rec_name"].isin(mis)]
print(f"D42 언급 128 CoT의 공개 기준 밖 추천({len(d42)}건) 중 등급 불일치 상품:", x3["cot_id"].nunique(), "CoT /", len(x3), "추천")
d42n = f2r[(f2r["성향_D21"] == "위험중립형") & ~f2r["3등급 언급"] & f2r["밖_공개기준"]]
x4 = d42n[d42n["rec_name"].isin(mis)]
print(f"D42 비언급 170 CoT의 공개 기준 밖 추천({len(d42n)}건) 중 등급 불일치 상품:", x4["cot_id"].nunique(), "CoT /", len(x4), "추천")

## H3. H1 의심 상품 무작위 20개 → `data/review/원천검수표.csv`

- 원문(설명)이 들어 있어 git 제외 폴더에 둔다. 이미 있으면 덮어쓰지 않는다.

In [ ]:
h3 = h1.sample(min(20, len(h1)), random_state=0)
review3 = pd.DataFrame({
    "상품명": h3["product_name"],
    "등급": h3["risk_grade"],
    "설명 앞부분": h3["investment_strategy"].fillna("").str.replace("\n", " ").str[:200],
    "내 판정": "",
})
out_path3 = ROOT / "data" / "review" / "원천검수표.csv"
if out_path3.exists():
    print("이미 있어 덮어쓰지 않음:", out_path3)
else:
    review3.to_csv(out_path3, index=False, encoding="utf-8-sig")
    print("저장:", out_path3, "/ 행 수:", len(review3))
print(review3[["상품명", "등급"]].to_string())

## H4. 교정검수표에 'cot2 안에 허용 범위·넓은 등급을 언급하는 문장 수' 열 추가

- 세는 문장 (cot2를 문장으로 나눈 뒤 하나라도 해당하면 1문장):
  - 허용 범위 표현: `parse_allowed_grades`가 그 문장에서 '범위·목록'을 찾음, 또는 문장에 '허용'과 '등급'이 함께 있음
  - 넓은 등급 언급: 위험중립형은 '3등급'·'다소 높은 위험', 안정추구형은 '4등급'·'보통 위험' (공백 무관)
- 애매한 경우 처리: 상품 설명 문장 속 등급 언급도 넓은 등급 언급으로 센다. 범위 표기 위치만 보려면 '허용 범위 문장 수' 열을 본다.
- 기존 파일의 `내 판정` 칸은 그대로 두고 열만 추가한다. 이미 열이 있으면 다시 쓰지 않는다.

In [ ]:
WIDE = {"위험중립형": r"3\s*등급|다소\s*높은\s*위험", "안정추구형": r"4\s*등급|보통\s*위험"}

def count_sents(cot2_text, p):
    n_range = n_any = 0
    for s in re.split(r"(?<=[.!?])\s+", str(cot2_text)):
        is_range = (parse_allowed_grades(s)[1].startswith("범위·목록")) or ("허용" in s and "등급" in s)
        is_wide = bool(re.search(WIDE.get(p, r"(?!)"), s))
        n_range += is_range
        n_any += is_range or is_wide
    return n_range, n_any

rv = pd.read_csv(out_path2, encoding="utf-8-sig", dtype=str, keep_default_na=False)
COL_ANY = "cot2 허용 범위·넓은 등급 언급 문장 수"
COL_RANGE = "cot2 허용 범위 문장 수"
if COL_ANY in rv.columns:
    print("이미 열이 있어 다시 쓰지 않음")
else:
    cot2_by_id = sc.set_index("cot_id")["cot2"]
    cnt = [count_sents(cot2_by_id[int(c)], p) for c, p in zip(rv["cot_id"], rv["성향"])]
    rv.insert(rv.columns.get_loc("내 판정"), COL_RANGE, [c[0] for c in cnt])
    rv.insert(rv.columns.get_loc("내 판정"), COL_ANY, [c[1] for c in cnt])
    try:
        rv.to_csv(out_path2, index=False, encoding="utf-8-sig")
        print("열 추가:", out_path2)
    except PermissionError:
        print("파일이 열려 있어 저장하지 못함 (엑셀을 닫고 이 셀을 다시 실행):", out_path2)
print("내 판정 채워진 행:", (rv["내 판정"] != "").sum())
print(rv[["cot_id", "성향", COL_RANGE, COL_ANY]].to_string())

## H5. 교정검수표에 '고객 위험 태도 문장'과 'cot2 전문' 열 추가

- 고객 위험 태도 문장: question을 문장으로 나눈 뒤, 성향 명칭(안정형~공격투자형) 또는 '손실'·'변동성'·'위험'·'안정'·'감수'·'원금'이 들어간 문장만 ` / `로 이어 붙인다.
  애매한 경우 처리: '안정적인 수익'처럼 위험 태도가 아닌 문장도 '안정'으로 잡힌다. 검수 때 읽고 거른다.
- cot2 전문: cot2 원문 그대로.
- 기존 `내 판정` 칸은 그대로 두고, 열이 이미 있으면 다시 쓰지 않는다.

In [ ]:
RISK_ATT = "|".join(PROPENSITY) + r"|손실|변동성|위험|안정|감수|원금"

def risk_attitude_sents(q):
    return " / ".join(s.strip() for s in re.split(r"(?<=[.!?])\s+", str(q)) if re.search(RISK_ATT, s))

rv = pd.read_csv(out_path2, encoding="utf-8-sig", dtype=str, keep_default_na=False)
before = rv["내 판정"].tolist()
q_by_id = sc.set_index("cot_id")["question"]
cot2_by_id = sc.set_index("cot_id")["cot2"]
added = []
for col, f in [("고객 위험 태도 문장", lambda c: risk_attitude_sents(q_by_id[int(c)])),
               ("cot2 전문", lambda c: str(cot2_by_id[int(c)]))]:
    if col not in rv.columns:
        rv.insert(rv.columns.get_loc("내 판정"), col, rv["cot_id"].map(f))
        added.append(col)
assert rv["내 판정"].tolist() == before   # 내 판정 칸이 바뀌지 않았는지 확인
if added:
    try:
        rv.to_csv(out_path2, index=False, encoding="utf-8-sig")
        print("열 추가:", added)
    except PermissionError:
        print("파일이 열려 있어 저장하지 못함 (엑셀을 닫고 이 셀을 다시 실행):", out_path2)
else:
    print("이미 열이 있어 다시 쓰지 않음")
print(rv.columns.tolist())
print("고객 위험 태도 문장이 빈 행:", (rv["고객 위험 태도 문장"] == "").sum())

## 02b 정리 (직접 작성)

1. 2번 등급 불일치는 어떤 패턴인가? 03 판정(기준 3)에서 상품 등급의 기준을 무엇으로 할 것인가? (cot3 표기 / JSON)
2. 3번에서 같은 성향인데 허용 등급이 다른 경우를 어떻게 볼 것인가?
3. 4·5·9번 불일치·미완결 건을 정제 데이터셋에서 어떻게 다룰 것인가?
4. 6번 고령자 고위험 추천을 품질 기준에 넣을 것인가? 근거 규정은 무엇인가? (docs/sources.md)
5. 7번 중복을 Train/Valid 구성과 정제 데이터셋에서 어떻게 처리할 것인가?
6. decisions.md에 추가할 항목:
